# 03 · TorchDistributor — M×N GPU

Multi-node 토폴로지(M×N)를 다룹니다. 단일 노드는 [02 · TorchDistributor — 1×1 / 1×N GPU](02-train_torch_distributor_1x1_1xN.ipynb)에서 다룹니다.

호출 형태는 다음과 같습니다.

| 섹션 | 모드 | 호출 |
|------|------|------|
| M×N  | DDP, multi node | `TorchDistributor(num_processes=M*N, local_mode=False).run(train_fn, ...)` |

데이터셋·모델·하이퍼파라미터는 1×1 / 1×N과 동일합니다(`CONFIG`, ML-25M). global batch만 `batch_size_per_gpu × M × N`으로 커집니다.

> `local_mode=False`는 **Spark worker 노드들에 child 프로세스를 분산 spawn**합니다. driver는 학습에 참여하지 않고 코디네이션만 담당하므로, driver-side `log_system_metrics`만 켜면 idle metric만 잡힙니다. single-node의 `local_mode=True`와의 비교는 [`concepts-torchdistributor-internals.md` — `local_mode`: driver vs worker](../00-foundations/concepts-torchdistributor-internals.md)를 참조합니다.

**클러스터**: Multi-node Classic GPU, DBR 17.3 LTS ML. Driver `g5.12xlarge` + Workers `g5.12xlarge` × M. **Autoscaling OFF 필수** (DDP는 시작 시 노드 수가 고정됩니다). Single node 토글 OFF. Serverless GPU는 multi-node 미지원이라 사용 불가입니다 ([env-databricks-environments.md](../00-foundations/env-databricks-environments.md)).

자주 만나는 함정 모음은 [`debug-common-pitfalls.md`](../00-foundations/debug-common-pitfalls.md), MLflow rank-0 로깅·child 자격증명 전달은 [`ops-mlflow-tracking.md`](../00-foundations/ops-mlflow-tracking.md)에 정리되어 있습니다.

In [ ]:
%run ./00-setup

## 학습 함수

TorchDistributor child가 실행할 entrypoint입니다. multi-node에서는 worker 프로세스가 다른 노드에서 fresh Python으로 시작하므로 **함수 내부에 import·클래스 정의를 모두 둡니다**. closure로 driver 변수를 잡지 말고 모두 인자로 전달합니다 ([debug-common-pitfalls.md §2](../00-foundations/debug-common-pitfalls.md)).

In [ ]:
def train_fn_ddp(
    run_id,
    db_host,
    db_token,
    data_dir,
    ckpt_path,
    n_users,
    n_items,
    emb_dim,
    tower_hidden,
    batch_size,
    num_epochs,
    max_steps_per_epoch,
    patience,
    min_delta,
    topology,
):
    """multi-node DDP. 1×N과 동일 시그니처. epoch 단위 + DDP val all_reduce + EarlyStopping."""
    import os

    import mlflow
    import pyarrow.parquet as pq
    import torch
    import torch.distributed as dist
    import torch.nn as nn
    from torch.nn.parallel import DistributedDataParallel as DDP
    from torch.utils.data import DataLoader, TensorDataset
    from torch.utils.data.distributed import DistributedSampler

    os.environ["DATABRICKS_HOST"] = db_host
    os.environ["DATABRICKS_TOKEN"] = db_token

    dist.init_process_group("nccl")
    local_rank = int(os.environ["LOCAL_RANK"])
    global_rank = int(os.environ["RANK"])
    world_size = int(os.environ["WORLD_SIZE"])
    torch.cuda.set_device(local_rank)
    device = torch.device("cuda", local_rank)

    class TwoTowerMLP(nn.Module):
        def __init__(self, n_users, n_items, emb_dim, tower_hidden):
            super().__init__()
            self.user_emb = nn.Embedding(n_users, emb_dim)
            self.item_emb = nn.Embedding(n_items, emb_dim)
            self.user_tower = self._make_tower(emb_dim, tower_hidden)
            self.item_tower = self._make_tower(emb_dim, tower_hidden)

        @staticmethod
        def _make_tower(in_dim, hidden):
            layers, prev = [], in_dim
            for h in hidden:
                layers += [nn.Linear(prev, h), nn.ReLU()]
                prev = h
            return nn.Sequential(*layers)

        def forward(self, user_ids, item_ids):
            u = self.user_tower(self.user_emb(user_ids))
            i = self.item_tower(self.item_emb(item_ids))
            return (u * i).sum(dim=-1)

    class EarlyStopping:
        def __init__(self, patience, min_delta):
            self.patience = patience
            self.min_delta = min_delta
            self.best = float("inf")
            self.counter = 0

        def step(self, val_loss):
            if val_loss < self.best - self.min_delta:
                self.best = val_loss
                self.counter = 0
                return False
            self.counter += 1
            return self.counter >= self.patience

    def load_split(split):
        split_dir = os.path.join(data_dir, split)
        files = sorted(
            os.path.join(split_dir, f)
            for f in os.listdir(split_dir)
            if f.endswith(".parquet")
        )
        table = pq.read_table(files, columns=["user_id", "item_id", "label"])
        return TensorDataset(
            torch.from_numpy(table.column("user_id").to_numpy()),
            torch.from_numpy(table.column("item_id").to_numpy()),
            torch.from_numpy(table.column("label").to_numpy()),
        )

    train_dataset = load_split("train")
    val_dataset = load_split("val")
    train_sampler = DistributedSampler(
        train_dataset, num_replicas=world_size, rank=global_rank, drop_last=True
    )
    val_sampler = DistributedSampler(
        val_dataset,
        num_replicas=world_size,
        rank=global_rank,
        shuffle=False,
        drop_last=False,
    )
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        sampler=train_sampler,
        num_workers=2,
        pin_memory=True,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        sampler=val_sampler,
        num_workers=2,
        pin_memory=True,
    )

    model = TwoTowerMLP(n_users, n_items, emb_dim, tower_hidden).to(device)
    model = DDP(model, device_ids=[local_rank], output_device=local_rank)
    loss_fn = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
    early_stop = EarlyStopping(patience=patience, min_delta=min_delta)

    if global_rank == 0:
        mlflow.start_run(run_id=run_id, log_system_metrics=True)
        mlflow.log_params(
            {
                "topology": topology,
                "world_size": world_size,
                "n_users": n_users,
                "n_items": n_items,
                "emb_dim": emb_dim,
                "batch_size": batch_size,
                "num_epochs": num_epochs,
                "max_steps_per_epoch": max_steps_per_epoch,
                "patience": patience,
                "min_delta": min_delta,
            }
        )

    global_step = 0
    for epoch in range(num_epochs):
        train_sampler.set_epoch(epoch)
        model.train()
        for step, (u, i, y) in enumerate(train_loader):
            if step >= max_steps_per_epoch:
                break
            u, i, y = u.to(device), i.to(device), y.to(device)
            optimizer.zero_grad(set_to_none=True)
            logits = model(u, i)
            loss = loss_fn(logits, y)
            loss.backward()
            optimizer.step()
            if global_rank == 0 and step % 20 == 0:
                mlflow.log_metric("train/loss", loss.item(), step=global_step)
            global_step += 1

        model.eval()
        local_loss_sum = torch.zeros(1, device=device)
        local_n = torch.zeros(1, device=device)
        with torch.no_grad():
            for u, i, y in val_loader:
                u, i, y = u.to(device), i.to(device), y.to(device)
                logits = model(u, i)
                loss = loss_fn(logits, y)
                local_loss_sum += loss * y.size(0)
                local_n += y.size(0)
        dist.all_reduce(local_loss_sum, op=dist.ReduceOp.SUM)
        dist.all_reduce(local_n, op=dist.ReduceOp.SUM)
        val_loss = (local_loss_sum / local_n).item()

        if global_rank == 0:
            mlflow.log_metric("val/loss", val_loss, step=epoch)
            print(
                f"[rank=0] epoch={epoch:2d} val_loss={val_loss:.6f} (best={early_stop.best:.6f} counter={early_stop.counter})"
            )

        if early_stop.step(val_loss):
            if global_rank == 0:
                print(
                    f"early stop at epoch {epoch} (best val_loss={early_stop.best:.6f})"
                )
                mlflow.log_metric("early_stop_epoch", epoch)
            break

    dist.barrier()
    if global_rank == 0:
        torch.save({"model": model.module.state_dict()}, ckpt_path)
        mlflow.log_param("ckpt_path", ckpt_path)
        mlflow.end_run()
        print(f"saved {ckpt_path}")
    dist.destroy_process_group()
    return "ok"

## M×N GPU

`local_mode=False`, `num_processes = M*N`으로 호출합니다. 1×1·1×N과 같은 `CONFIG`·같은 데이터셋(`DATA_DIR`)을 사용합니다.

multi-node 실행 시 다음 세 가지를 함께 점검해 두면 좋습니다.

- DataLoader가 각 rank에서 동일한 parquet 디렉토리를 메모리에 로드합니다. ML-25M train ~600MB 정도면 노드당 메모리는 충분합니다.
- inter-node bandwidth(EFA)가 없으면 throughput이 1×N과 큰 차이 없을 수 있습니다 ([debug-common-pitfalls.md §10](../00-foundations/debug-common-pitfalls.md)).
- early stopping은 모든 rank가 동일한 `EarlyStopping.step` 결과로 break합니다 (val_loss를 `dist.all_reduce`로 합산).

In [ ]:
import os

import mlflow
from pyspark.ml.torch.distributor import TorchDistributor

NUM_NODES = 2  # M (worker 노드 수. driver는 학습에 참여하지 않음 — local_mode=False)
NUM_GPUS_PER_NODE = 4  # N

cfg = CONFIG
mlflow.set_experiment(EXPERIMENT_PATH)
with mlflow.start_run(run_name="MxN", log_system_metrics=True) as run:
    run_id = run.info.run_id

TorchDistributor(
    num_processes=NUM_NODES * NUM_GPUS_PER_NODE,
    local_mode=False,
    use_gpu=True,
).run(
    train_fn_ddp,
    run_id=run_id,
    db_host=DB_HOST,
    db_token=DB_TOKEN,
    data_dir=DATA_DIR,
    ckpt_path=os.path.join(CKPT_DIR, "MxN.pt"),
    n_users=cfg["n_users"],
    n_items=cfg["n_items"],
    emb_dim=cfg["emb_dim"],
    tower_hidden=cfg["tower_hidden"],
    batch_size=cfg["batch_size_per_gpu"],
    num_epochs=cfg["num_epochs"],
    max_steps_per_epoch=cfg["max_steps_per_epoch"],
    patience=PATIENCE,
    min_delta=MIN_DELTA,
    topology="MxN",
)